##### Script 1: RPF Fix (run on ALL nodes first)

In [ ]:
cat > fix-rpf.sh <<OF
#!/bin/bash
# =============================================================================
# RPF Fix Script — Run this on ALL nodes before installing Calico
# Fixes: strict Reverse Path Filtering that drops pod return traffic
# Run on: master-1, master-2, master-3, worker-1, worker-2
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

IFACE="${1:-ens192}"  # Pass interface name as argument if different, e.g. ./fix-rpf.sh eth0

log "Setting loose RPF (rp_filter=2) on interface: $IFACE"

# Apply immediately
sysctl -w net.ipv4.conf.all.rp_filter=2
sysctl -w net.ipv4.conf.default.rp_filter=2
sysctl -w net.ipv4.conf.${IFACE}.rp_filter=2

# Make permanent
cat > /etc/sysctl.d/99-kubernetes.conf << EOF
# Required for Kubernetes + Calico VXLAN + hostNetwork ingress
net.ipv4.conf.all.rp_filter = 2
net.ipv4.conf.default.rp_filter = 2
net.ipv4.conf.${IFACE}.rp_filter = 2

# Required for Kubernetes
net.bridge.bridge-nf-call-iptables = 1
net.bridge.bridge-nf-call-ip6tables = 1
net.ipv4.ip_forward = 1
EOF

sysctl -p /etc/sysctl.d/99-kubernetes.conf

success "RPF fix applied permanently on $(hostname)"
echo "Current rp_filter values:"
echo "  all:     $(cat /proc/sys/net/ipv4/conf/all/rp_filter)"
echo "  default: $(cat /proc/sys/net/ipv4/conf/default/rp_filter)"
echo "  ${IFACE}: $(cat /proc/sys/net/ipv4/conf/${IFACE}/rp_filter)"
OF

In [ ]:
chmod +x fix-rpf.sh && ./fix-rpf.sh ens192

---

##### Script 2: Full Cleanup

In [ ]:
cat > cleanup.sh <<'OOF'
#!/bin/bash

# =============================================================================
# ========================== Full Cleanup Script ==============================
# ========================== Removes: Headlamp, Ingress, MetalLB ==============
# =============================================================================
set -euo pipefail

RED='\033[0;31m'; 
GREEN='\033[0;32m'; 
YELLOW='\033[1;33m'; 
BLUE='\033[0;34m'; 
NC='\033[0m'

log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }
warn()    { echo -e "${YELLOW}[WARN]${NC} $1"; }

# ────────────────────────── Reusable Force Finalize Function ──────────────────────────
# Forcefully cleans up any namespace stuck in 'Terminating' due to API discovery bugs
force_cleanup_namespace() {
    local ns=$1
    sleep 2
    
    if kubectl get namespace "$ns" &>/dev/null; then
        local status
        status=$(kubectl get namespace "$ns" -o jsonpath='{.status.phase}' 2>/dev/null || echo "")
        
        if [ "$status" == "Terminating" ]; then
            warn "Namespace '$ns' is stuck in Terminating. Forcing finalizer removal..."
            
            # Start background proxy on an isolated high port
            local proxy_port=8099
            kubectl proxy --port=$proxy_port &
            local proxy_pid=$!
            
            sleep 2
            
            # Extract, strip, and PUT back via proxy bypass
            kubectl get namespace "$ns" -o json \
              | sed 's/"kubernetes"//g' > "tmp_${ns}_finalize.json"
              
            curl -s -k -H "Content-Type: application/json" \
                 -X PUT --data-binary @"tmp_${ns}_finalize.json" \
                 http://127.0.0.1:$proxy_port/api/v1/namespaces/$ns/finalize > /dev/null
            
            rm -f "tmp_${ns}_finalize.json"
            kill $proxy_pid &>/dev/null || true
            success "Forcefully removed finalizers for $ns"
        fi
    fi
}

# ────────────────────────── 1. Headlamp ──────────────────────────
log "1. Removing Headlamp..."
helm uninstall my-headlamp -n headlamp 2>/dev/null || warn "Headlamp not found"
kubectl delete namespace headlamp --ignore-not-found --wait=false
force_cleanup_namespace "headlamp"
success "Headlamp removed"

# ────────────────────────── 2. Ingress-NGINX ──────────────────────────
log "2. Removing Ingress-NGINX..."
helm uninstall ingress-nginx -n ingress-nginx 2>/dev/null || warn "ingress-nginx not found"
kubectl delete namespace ingress-nginx --ignore-not-found --wait=false
force_cleanup_namespace "ingress-nginx"
success "✅ Ingress-NGINX removed completely"

# ────────────────────────── 3. MetalLB ──────────────────────────
log "3. Removing MetalLB..."
kubectl delete -f metallb-pool.yaml --ignore-not-found 2>/dev/null || true
kubectl delete -f https://raw.githubusercontent.com/metallb/metallb/v0.16.0/config/manifests/metallb-native.yaml --ignore-not-found --wait=false 2>/dev/null || true

# Delete namespace asynchronously so it doesn't block execution
kubectl delete namespace metallb-system --ignore-not-found --wait=false
force_cleanup_namespace "metallb-system"
success "✅ MetalLB removed"

# ────────────────────────── 4. Calico ──────────────────────────
echo "=== Starting Complete Calico & Tigera Purge ==="

# 1. Delete the installation configuration to stop the operator
kubectl delete installation default --timeout=15s || true
kubectl delete apiserver default --timeout=15s || true

# 2. Forcefully patch and remove finalizers from stuck operator resources
kubectl patch installation default --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true
kubectl patch apiserver default --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true

# 3. Delete the deployments and operators
kubectl delete -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml --ignore-not-found=true || true
kubectl delete ns calico-system tigera-operator --timeout=30s || true

# 4. Force-remove namespaces if they are stuck in 'Terminating'
for ns in calico-system tigera-operator; do
    if kubectl get ns $ns &>/dev/null; then
        echo "Namespace $ns is stuck. Removing finalizers..."
        kubectl get namespace $ns -o json | tr -d "\n" | sed 's/"finalizers": \[[^]]*\]/"finalizers": []/' | kubectl replace --raw /api/v1/namespaces/$ns/finalize -f - || true
    fi
done

# 5. Clean up ALL Calico/Tigera CRDs completely
echo "Cleaning up CRDs..."
kubectl get crd -o name | grep -E '(tigera|calico)' | xargs kubectl delete --timeout=15s || true

# 6. Force remove finalizers from any lingering Calico CRDs
for crd in $(kubectl get crd -o name | grep -E '(tigera|calico)'); do
    kubectl patch $crd --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true
    kubectl delete $crd || true
done

# 7. Clean up any lingering Felix configuration clusterwide
kubectl delete clusterrolebinding tigera-operator || true
kubectl delete clusterrole tigera-operator || true

echo "=== Purge Complete! Your cluster is clean. ==="

success "✅ Calico removed"

# ────────────────────────── 5. Reset kube-proxy to iptables mode ──────────────────────────
log "5. Resetting kube-proxy configmap to default iptables mode..."
kubectl get configmap kube-proxy -n kube-system -o yaml | \
  sed 's/mode: "ipvs"/mode: "iptables"/' | \
  kubectl apply -f - 2>/dev/null || warn "Could not reset kube-proxy config"
kubectl rollout restart daemonset/kube-proxy -n kube-system
success "✅ kube-proxy reset"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Cleanup complete. Wait 30s before running install scripts."
echo "═══════════════════════════════════════════════════════════"
OOF

In [ ]:
chmod +x cleanup.sh && ./cleanup.sh

---

##### Script 3: Install Calico

In [ ]:
cat > install-calico.sh << 'OF'
#!/bin/bash
# =============================================================================
# Calico Install Script — v3.32.0 via Tigera Operator (Idempotent Version)
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m';
BLUE='\033[0;34m'; 
YELLOW='\033[1;33m';
NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }
warn()    { echo -e "${YELLOW}[WARN]${NC} $1"; }

POD_CIDR="10.244.0.0/16"

# ────────────────────── 1. kube-proxy: IPVS + strictARP ──────────────────────

log "1. Configuring kube-proxy for IPVS + strictARP..."

kubectl get configmap kube-proxy -n kube-system -o yaml | \
  sed 's/mode: ""/mode: "ipvs"/' | \
  sed 's/strictARP: false/strictARP: true/' | \
  kubectl apply -f -

kubectl rollout restart daemonset/kube-proxy -n kube-system
kubectl rollout status daemonset/kube-proxy -n kube-system

success "✅ kube-proxy configured"

# ────────────────────── 2. Install Tigera Operator ──────────────────────

log "2. Installing Tigera operator..."

kubectl apply -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml
log "Waiting for Tigera operator to be ready..."
kubectl rollout status deployment/tigera-operator -n tigera-operator

success "✅ Tigera operator ready"

# ────────────────────── 3. Wait for CRDs ──────────────────────

log "3. Waiting for Calico CRDs to be registered..."

# Tweak: Loop checks every 5 seconds instead of 20, checking the raw CRD list
until kubectl get crd | grep -q "installations.operator.tigera.io"; do
  echo "   CRD not ready yet, waiting 5s..."
  sleep 5
done

success "✅ Calico CRDs registered"

# ────────────────────── 4. Apply Installation CR ──────────────────────

log "4. Applying Calico installation with VXLAN + MTU 1450..."

cat << YAML | kubectl apply -f -
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  variant: Calico
  calicoNetwork:
    mtu: 1450
    bgp: Disabled
    ipPools:
    - name: default-ipv4-ippool
      cidr: ${POD_CIDR}
      blockSize: 26
      encapsulation: VXLAN
      natOutgoing: Enabled
      nodeSelector: all()
---
apiVersion: operator.tigera.io/v1
kind: APIServer
metadata:
  name: default
spec: {}
YAML

success "✅ Installation CR applied"

# ────────────────────── 5. Wait for calico-system namespace ──────────────────────
log "5. Waiting for calico-system namespace to be created by operator..."

# Tweak: Loop every 5 seconds instead of 150
until kubectl get namespace calico-system &>/dev/null; do
  echo "   Namespace not ready yet, waiting 5s..."
  sleep 5
done
success "✅ calico-system namespace ready"

# ────────────────────── 6. Wait for Calico nodes ──────────────────────

log "6. Waiting for Calico nodes to be ready..."

# Tweak: Loop every 5 seconds instead of 150
until kubectl get daemonset calico-node -n calico-system &>/dev/null; do
  echo "   Daemonset not created yet, waiting 5s..."
  sleep 5
done

kubectl rollout status daemonset/calico-node -n calico-system
success "✅ Calico nodes ready"

log "6.1 Waiting for Calico API server..."
kubectl rollout status deployment/calico-apiserver -n calico-system

# ────────────────────── 7. Apply Felix configuration ──────────────────────
log "7. Applying Felix configuration..."

# Tweak: Wrap Felix in a quick loop in case the Felix CRD is still registering 
until kubectl get crd felixconfigurations.crd.projectcalico.org &>/dev/null; do
  echo "   Waiting for Felix CRD to be registered by the API Server..."
  sleep 5
done

cat << YAML | kubectl apply -f -
apiVersion: crd.projectcalico.org/v1
kind: FelixConfiguration
metadata:
  name: default
spec:
  genericXDPEnabled: false
  natPortRange: "32768:65535"
YAML
success "✅ Felix configuration applied"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Calico configured/verified successfully"
echo "   Pod CIDR : ${POD_CIDR}"
echo "   MTU      : 1450"
echo "   Mode     : VXLAN"
echo "═══════════════════════════════════════════════════════════"

kubectl get tigerastatus
OF

In [ ]:
chmod +x install-calico.sh && ./install-calico.sh

For delete 

In [ ]:
cat > purge-calico.sh << 'EOF'
#!/bin/bash
set -x

echo "=== Starting Complete Calico & Tigera Purge ==="

# 1. Delete the installation configuration to stop the operator
kubectl delete installation default --timeout=15s || true
kubectl delete apiserver default --timeout=15s || true

# 2. Forcefully patch and remove finalizers from stuck operator resources
kubectl patch installation default --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true
kubectl patch apiserver default --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true

# 3. Delete the deployments and operators
kubectl delete -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml --ignore-not-found=true || true
kubectl delete ns calico-system tigera-operator --timeout=30s || true

# 4. Force-remove namespaces if they are stuck in 'Terminating'
for ns in calico-system tigera-operator; do
    if kubectl get ns $ns &>/dev/null; then
        echo "Namespace $ns is stuck. Removing finalizers..."
        kubectl get namespace $ns -o json | tr -d "\n" | sed 's/"finalizers": \[[^]]*\]/"finalizers": []/' | kubectl replace --raw /api/v1/namespaces/$ns/finalize -f - || true
    fi
done

# 5. Clean up ALL Calico/Tigera CRDs completely
echo "Cleaning up CRDs..."
kubectl get crd -o name | grep -E '(tigera|calico)' | xargs kubectl delete --timeout=15s || true

# 6. Force remove finalizers from any lingering Calico CRDs
for crd in $(kubectl get crd -o name | grep -E '(tigera|calico)'); do
    kubectl patch $crd --type json -p '[{"op": "remove", "path": "/metadata/finalizers"}]' || true
    kubectl delete $crd || true
done

# 7. Clean up any lingering Felix configuration clusterwide
kubectl delete clusterrolebinding tigera-operator || true
kubectl delete clusterrole tigera-operator || true

echo "=== Purge Complete! Your cluster is clean. ==="
EOF

In [ ]:
chmod +x purge-calico.sh && ./purge-calico.sh

Install Fennal

In [ ]:
cat > install-flannel.sh << 'OF'
#!/bin/bash
# =============================================================================
# Flannel CNI Install Script
# Replaces Calico — works cleanly with kube-vip + MetalLB + hostNetwork ingress
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; YELLOW='\033[1;33m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }
warn()    { echo -e "${YELLOW}[WARN]${NC} $1"; }

POD_CIDR="10.244.0.0/16"

# ────────────────── 1. kube-proxy: IPVS + strictARP ──────────────────
log "1. Configuring kube-proxy for IPVS + strictARP..."
kubectl get configmap kube-proxy -n kube-system -o yaml | \
  sed 's/mode: ""/mode: "ipvs"/' | \
  sed 's/strictARP: false/strictARP: true/' | \
  kubectl apply -f -
kubectl rollout restart daemonset/kube-proxy -n kube-system
kubectl rollout status daemonset/kube-proxy -n kube-system 
success "✅ kube-proxy configured"

# ────────────────── 2. Install Flannel ──────────────────
log "2. Installing Flannel..."
kubectl apply -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml

success "✅ Install Flannel successfully"

# ────────────────── 3. Wait for Flannel pods ──────────────────
log "3. Waiting for kube-flannel namespace..."
until kubectl get namespace kube-flannel &>/dev/null; do
  echo "  waiting..."; sleep 3
done

success "✅ kube-flannel namespace create successfully"

log "3.1 Waiting for Flannel DaemonSet to be ready..."
until kubectl get daemonset kube-flannel-ds -n kube-flannel &>/dev/null; do
  echo "  daemonset not created yet, waiting 5s..."; sleep 5
done
kubectl rollout status daemonset/kube-flannel-ds -n kube-flannel 

success "✅ Flannel ready"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Flannel installed successfully"
echo "   Pod CIDR : ${POD_CIDR}"
echo "   Mode     : VXLAN (default)"
echo "═══════════════════════════════════════════════════════════"
echo "Run: kubectl get pods -n kube-flannel"
OF

In [ ]:
chmod +x install-flannel.sh && ./install-flannel.sh

In [ ]:
kubectl rollout restart daemonset/ingress-nginx-controller -n ingress-nginx
kubectl rollout status daemonset/ingress-nginx-controller -n ingress-nginx

For delelte

In [ ]:
# 1. Delete the flannel daemonset and resources
kubectl delete -f https://github.com/flannel-io/flannel/blob/master/Documentation/kube-flannel.yml --ignore-not-found=true

# 2. Double check and delete any leftover flannel resources manually if needed
kubectl delete daemonset kube-flannel-ds -n kube-flannel --ignore-not-found=true
kubectl delete namespace kube-flannel --ignore-not-found=true

---

---

##### Script 4: Install MetalLB

In [ ]:
cat > install-metallb.sh << 'OF'
#!/bin/bash
# =============================================================================
# MetalLB Install Script — v0.16.0
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

LB_RANGE="172.16.6.90-172.16.6.95"

# ────────────────────────── 1. Install MetalLB ──────────────────────────
log "1. Installing MetalLB v0.16.0..."
kubectl apply -f https://raw.githubusercontent.com/metallb/metallb/v0.16.0/config/manifests/metallb-native.yaml

# ────────────────────────── 2. Remove webhook (avoids timing issues on fresh install) ─────────────────
log "2. Removing MetalLB webhook..."

kubectl delete validatingwebhookconfiguration metallb-webhook-configuration --ignore-not-found

sleep 10

success "✅ Removing MetalLB webhook"

# ────────────────────────── 

log "2.1 Waiting for MetalLB controller..."

kubectl rollout status deployment/controller -n metallb-system 
kubectl rollout status daemonset/speaker -n metallb-system 

success "✅ MetalLB ready"


# ────────────────────────── 3. Apply IP pool + L2 advertisement ──────────────────────────────────────

log "3. Configuring IP address pool: ${LB_RANGE}..."

cat << EOF | kubectl apply -f -
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: first-pool
  namespace: metallb-system
spec:
  addresses:
  - ${LB_RANGE}
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: l2-adv
  namespace: metallb-system
spec:
  ipAddressPools:
  - first-pool
EOF

success "✅ MetalLB IP pool configured"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ MetalLB installed successfully"
echo "   IP Range : ${LB_RANGE}"
echo "═══════════════════════════════════════════════════════════"

kubectl get pods -n metallb-system
OF

In [ ]:
chmod +x install-metallb.sh && ./install-metallb.sh 

---

##### Script 5: Install Ingress-NGINX

In [ ]:
cat > install-ingress.sh << 'OF'
#!/bin/bash
# =============================================================================
# Ingress-NGINX Install Script
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

LB_IP="172.16.6.90"

# ────────────────────────── 1. Add Helm repo ──────────────────────────
log "1. Adding ingress-nginx Helm repo..."

helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

success "✅ Add ingress-nginx Helm successfully"

# ────────────────────────── 2. Install ──────────────────────────

log "2. Installing ingress-nginx with DaemonSet + hostNetwork..."

cat << EOF > /tmp/nginx-ingress-values.yaml
controller:
  kind: DaemonSet
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet
  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443
  service:
    type: LoadBalancer
    loadBalancerIP: ${LB_IP}
  admissionWebhooks:
    enabled: false
  metrics:
    enabled: true
  podDisruptionBudget:
    enabled: true
  tolerations:
  - key: "node-role.kubernetes.io/control-plane"
    operator: "Exists"
    effect: "NoSchedule"
  config:
    proxy-connect-timeout: "60"
    proxy-read-timeout: "60"
    proxy-send-timeout: "60"
EOF

helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f /tmp/nginx-ingress-values.yaml
 
 success "✅ Installing Ingress-NGINX successfully"

# ────────────────────────── 3. Waiting for ingress-nginx DaemonSet ──────────────────────────

log "3. Waiting for ingress-nginx DaemonSet..."

kubectl rollout status daemonset/ingress-nginx-controller -n ingress-nginx

success "✅ Ingress-NGINX ready"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Ingress-NGINX installed successfully"
echo "   LoadBalancer IP : ${LB_IP}"
echo "═══════════════════════════════════════════════════════════"

kubectl get svc -n ingress-nginx
OF

In [ ]:
chmod +x install-ingress.sh && ./install-ingress.sh

---

##### Script 6: Install Headlamp

In [ ]:
cat > install-headlamp.sh << 'OF'
#!/bin/bash
# =============================================================================
# Headlamp Install Script
# Fix included: ClusterRoleBinding so headlamp can reach the API server
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

DOMAIN="headlamp.voip.local"

# ────────────────────────── 1. Add Helm repo ──────────────────────────
log "1. Adding headlamp Helm repo..."

helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

success "✅ Added headlamp Helm successfully"

# ────────────────────────── 2. Install Headlamp ──────────────────────────
log "2. Installing Headlamp..."

cat << EOF > /tmp/headlamp-values.yaml
ingress:
  enabled: true
  ingressClassName: nginx
  annotations:
    nginx.ingress.kubernetes.io/ssl-redirect: "false"
    nginx.ingress.kubernetes.io/proxy-connect-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "60"
  hosts:
    - host: ${DOMAIN}
      paths:
        - path: /
          type: Prefix
EOF

helm upgrade --install my-headlamp headlamp/headlamp \
  -f /tmp/headlamp-values.yaml \
  --namespace headlamp \
  --create-namespace

success "✅ Installing Headlamp successfully"

# ────────────────────────── 3. Fix RBAC — this was missing and caused 504 ──────────────────────────
log "3. Applying ClusterRoleBinding for headlamp service account..."

kubectl create clusterrolebinding headlamp-admin \
  --clusterrole=cluster-admin \
  --serviceaccount=headlamp:my-headlamp \
  --dry-run=client -o yaml | kubectl apply -f -

success "✅ RBAC configured"

# ────────────────────────── 4. Wait and verify ──────────────────────────
log "4. Waiting for headlamp pod..."

kubectl rollout status deployment/my-headlamp -n headlamp 

success "✅ Headlamp ready"

# ────────────────────────── 5. Generate access token ──────────────────────────
echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Headlamp installed successfully"
echo "   URL    : http://${DOMAIN}"
echo "═══════════════════════════════════════════════════════════"
echo ""
echo "🔑 Access token (save this):"
kubectl create token my-headlamp --namespace headlamp
echo ""
echo "Test access:"

curl -H 'Host: ${DOMAIN}' http://172.16.6.90
OF

In [ ]:
chmod +x install-headlamp.sh  && ./install-headlamp.sh 